### Baseline Model

Based on the EDA.ipynb we can safely start with a baseline of a yes/no model

- Input: Image and Question
- Output: Probability of yes or no

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import torch.nn as nn
import random
import numpy as np

from transformers import AutoTokenizer, AutoModel

from datasets import load_dataset

In [ ]:
def print_device_info():
    print("PyTorch version:", torch.__version__)
    if torch.cuda.is_available():
        print("CUDA available:", torch.cuda.is_available())
        print("Number of GPUs:", torch.cuda.device_count())
        print("Current device index:", torch.cuda.current_device())
        print("Current device name:", torch.cuda.get_device_name(0))
        device = torch.device("cuda")
    else:
        print("CUDA not available, using CPU.")
        device = torch.device("cpu")
    return device

device = print_device_info()

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

In [ ]:
ds = load_dataset("flaviagiammarino/vqa-rad")
train = ds["train"]
test = ds["test"]

def yes_no(example):
    return example["answer"].lower() in ["yes", "no"]

train_yn = train.filter(yes_no)
test_yn = test.filter(yes_no)

# Check the length of the datasets
print(len(train_yn), len(test_yn))

The number of Yes and No in the train and test are accurate

In [ ]:
label_map = {"yes": 1, "no": 0}
train_yn = train_yn.map(lambda ex: {"label": label_map[ex["answer"].lower()]})
test_yn = test_yn.map(lambda ex: {"label": label_map[ex["answer"].lower()]})

Based on EDA
- Widths cluster roughly around 500–600 px and around 1000 px, with a few very large outliers
- Heights cluster around 400–800 px, with some up to around 1400 px

for the Baseline 224x224 is a standard size used by most CNNs such as ResNet

Downsizing from larger radiology images may:
- Loses some fine detail but for first experiments it is common to keep the compute cheap by keeping it to 224x224

We could be more careful by adding a CenterCrop

In [ ]:
# Img_tranform with CenterCrop and Resize
img_transform = transforms.Compose([
    transforms.Resize(512),  # resize shortest side to 512 keeping the aspect ratio
    transforms.CenterCrop(512),  
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [ ]:
# text
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
text_model = AutoModel.from_pretrained("distilbert-base-uncased")
text_model.eval() # Freezing this for baseline
for p in text_model.parameters():
    p.requires_grad = False

In [ ]:
class VQARADBinaryImageText(Dataset):
    def __init__(self, hf_split, img_tranform, tokenizer, text_model, max_length=32):
        self.data = hf_split
        self.img_transform = img_tranform
        self.tokenizer = tokenizer
        self.text_model = text_model
        self.max_length = max_length # Based on EDA most of the questions are short so for Baseline 32 should be good 

    def __len__(self):
        return len(self.data)

    # Tokenize the question and get the CLS token embedding as the question representation
    def encode_question(self, question):
        inputs = self.tokenizer(question,
                                return_tensors="pt",
                                truncation=True,
                                padding="max_length",
                                max_length=self.max_length)
        with torch.no_grad():
            outputs = self.text_model(**inputs)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # (1, hidden_size)
        return cls_embedding.squeeze(0)  # (hidden_size,)

    def __getitem__(self, idx):
        ex = self.data[idx]
        image = ex["image"]
        if self.img_transform:
            img_tensor = self.img_transform(image)  # (3, 224, 224)
        
        question = ex["question"]
        text_embedding = self.encode_question(question)  # (hidden_dim,)

        label = ex["label"]  # 0 or 1
        return img_tensor, text_embedding, label

In [ ]:
# To check if the dataset class is working correctly
dataset = VQARADBinaryImageText(train_yn, img_transform, tokenizer, text_model)
img, text_emb, label = dataset[3]
print("img shape:", img.shape)
print("text_emb shape:", text_emb.shape)
print("label:", label)

Utilizing the computing power of my PC at home

In [ ]:
train_dataset = VQARADBinaryImageText(train_yn, img_transform, tokenizer, text_model)
test_dataset = VQARADBinaryImageText(test_yn, img_transform, tokenizer, text_model)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

Build CNN and a MLP over concatenated features

In [ ]:
class ImageTextBinaryModel(nn.Module):
    def __init__(self, text_embedding_dim=768):
        """text_embedding_dim is the dimension of the text features from DistilBERT (768 for distilbert-base-uncased)"""
        super().__init__()
        # Image Branch of the model
        self.cnn = nn.Sequential(
        nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2))

        # After two max-pool layers, the feature map size will be (32, 56, 56) for 224x224 input
        self.img_feature_dim = 32 * 56 * 56  # After two max-pool layers on 224x224 input

        # Fusion and classification layers
        self.classifier = nn.Sequential(
            nn.Linear(self.img_feature_dim + text_embedding_dim, 256), # Fusion of image and text features
            nn.ReLU(),
            nn.Linear(256, 1)  # Binary classification
        )

    def forward(self, images, text_embeddings):
        """images: (B, 3, 224, 224), text_embeddings: (B, text_embedding_dim) and returns logits of shape (B,)"""
        x_img = self.cnn(images) # where images is (B, 3, 224, 224) B is batch size
        x_img = x_img.view(x_img.size(0), -1) # Flatten to (B, img_feature_dim) where img_feature_dim is 32*56*56
        x = torch.cat((x_img, text_embeddings), dim=1) # Concatenate image and text features (B, img_feature_dim + text_embedding_dim)
        logits = self.classifier(x) # (B, 1) where 1 is for binary classification
        return logits.squeeze(1) # Return (B,) for binary classification

In [ ]:
# Training Loop
model = ImageTextBinaryModel(text_embedding_dim=768).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(3):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, text_embeddings, labels in train_loader:
        logits = model(images.to(device), text_embeddings.to(device))
        loss = criterion(logits, labels.float().to(device))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(logits) > 0.5).long()
        correct += (preds.cpu() == labels).sum().item()
        total += labels.size(0)

print(f"Epoch {epoch+1}, Loss: {running_loss/total:.4f}, Accuracy: {correct/total:.4f}")

In [ ]:
# After training, evaluate on the test set
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, text_embeddings, labels in test_loader:
        logits = model(images.to(device), text_embeddings.to(device))
        preds = (torch.sigmoid(logits) > 0.5).long()
        correct += (preds.cpu() == labels).sum().item()
        total += labels.size(0)
print(f"Test Accuracy: {correct/total:.4f}")

In [ ]:
torch.save(model.state_dict(), "data\\04-models\\baseline_image_text_binary.pt")